In [0]:
%pip install python-jobspy pandas tqdm --ignore-requires-python --no-deps
%pip install requests beautifulsoup4 markdownify pydantic tls-client regex

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
"""
LinkedIn Tech Jobs Scraper
========================
Scrapes tech job listings from LinkedIn across major Arab and foreign countries.
Handles pagination, retries, deduplication, and checkpointing automatically.

Output: linkedin_jobs.csv

Usage:
    pip install python-jobspy pandas tqdm
    python linkedin_scraper.py
"""

import pandas as pd
import time
import random
import os
from tqdm import tqdm

try:
    from jobspy import scrape_jobs
except ImportError:
    print("ERROR: jobspy is not installed. Run:")
    print("   pip install python-jobspy")
    exit(1)


# ─────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────

# Job titles to search for across all locations
KEYWORDS = [
    # Development
    "software engineer",
    "frontend developer",
    "backend developer",
    "full stack developer",
    "mobile developer",
    "ios developer",
    "android developer",
    "web developer",
    # Data & AI
    "data scientist",
    "data analyst",
    "data engineer",
    "machine learning engineer",
    "AI engineer",
    "business intelligence",
    "nlp engineer",
    "computer vision engineer",
    # Infrastructure & Security
    "devops engineer",
    "cloud engineer",
    "site reliability engineer",
    "cybersecurity engineer",
    "network engineer",
    "linux administrator",
    # Product & Design
    "product manager",
    "product designer",
    "UX designer",
    "UI designer",
    "UX researcher",
    # Other Tech
    "QA engineer",
    "blockchain developer",
    "game developer",
]

# Target countries — key is the display name, value is the LinkedIn location string
LOCATIONS = {
    # Arab countries
    "Egypt":          "Egypt",
    "Saudi Arabia":   "Saudi Arabia",
    "UAE":            "United Arab Emirates",
    "Qatar":          "Qatar",
    "Kuwait":         "Kuwait",
}

# Number of results to fetch per keyword/location combination
RESULTS_PER_SEARCH = 20

# Random delay between requests to avoid rate limiting (in seconds)
DELAY_MIN = 8
DELAY_MAX = 15

OUTPUT_FILE     = "/Workspace/Users/felooamer@gmail.com/linkedin_jobs.csv"
CHECKPOINT_FILE = "/Workspace/Users/felooamer@gmail.com/linkedin_checkpoint.csv"  


# ─────────────────────────────────────────────
# Columns to keep in the final CSV
# ─────────────────────────────────────────────

COLUMNS = [
    "title",       # Job title
    "company",     # Company name
    "location",    # City / region
    "country",     # Country (added by us)
    "date_posted", # Posting date
    "job_type",    # Full-time, part-time, remote, etc.
    "job_url",     # Direct link to the job posting
]


# ─────────────────────────────────────────────
# Helper functions
# ─────────────────────────────────────────────

def load_checkpoint():
    """Load previously saved progress so we can resume if the script was interrupted."""
    if os.path.exists(CHECKPOINT_FILE):
        print("Checkpoint found. Resuming from last saved point...")
        return pd.read_csv(CHECKPOINT_FILE)
    return pd.DataFrame()


def save_checkpoint(df):
    """Save current progress to a temporary CSV file."""
    df.to_csv(CHECKPOINT_FILE, index=False, encoding="utf-8-sig")


def clean_dataframe(df, country_name):
    """
    Normalize and clean a scraped dataframe:
    - Keep only the columns we need
    - Add the country column
    - Standardize date format
    - Strip whitespace from text fields
    """
    # Keep only available columns from our list
    available = [c for c in COLUMNS if c in df.columns]
    df = df[available].copy()

    # Tag each row with the country it was scraped from
    df["country"] = country_name

    # Standardize date to YYYY-MM-DD
    if "date_posted" in df.columns:
        df["date_posted"] = pd.to_datetime(
            df["date_posted"], errors="coerce"
        ).dt.strftime("%Y-%m-%d")

    # Strip extra whitespace from text columns
    for col in ["title", "company", "location"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

    return df


def scrape_with_retry(keyword, location_name, retries=3):
    """
    Scrape LinkedIn for a given keyword and location.
    Retries up to 3 times with increasing wait time on failure.
    Returns an empty DataFrame if all attempts fail.
    """
    for attempt in range(1, retries + 1):
        try:
            jobs = scrape_jobs(
                site_name=["linkedin"],
                search_term=keyword,
                location=location_name,
                results_wanted=RESULTS_PER_SEARCH,
            )
            return jobs
        except Exception as e:
            wait = attempt * 10
            print(f"      Attempt {attempt}/{retries} failed: {e}")
            if attempt < retries:
                print(f"      Waiting {wait} seconds before retry...")
                time.sleep(wait)

    # Return empty DataFrame if all retries failed
    return pd.DataFrame()


# ─────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────

def main():
    print("=" * 55)
    print("  LinkedIn Tech Jobs Scraper")
    print(f"  {len(KEYWORDS)} keywords x {len(LOCATIONS)} locations")
    print(f"  Total searches: {len(KEYWORDS) * len(LOCATIONS)}")
    print("=" * 55)

    # Load checkpoint data if available
    all_jobs = load_checkpoint()

    # Track already seen URLs to avoid duplicates across searches
    seen_urls = set(all_jobs["job_url"].tolist()) if not all_jobs.empty else set()

    total_searches = len(KEYWORDS) * len(LOCATIONS)
    search_count   = 0
    new_jobs_total = 0

    # Progress bar to track overall scraping progress
    pbar = tqdm(total=total_searches, desc="Overall Progress", unit="search")

    for location_name, country_code in LOCATIONS.items():
        for keyword in KEYWORDS:
            search_count += 1

            # Update progress bar label
            pbar.set_postfix({
                "location": location_name[:10],
                "keyword":  keyword[:15],
                "total":    len(all_jobs),
            })

            # Scrape jobs for this keyword + location
            df = scrape_with_retry(keyword, location_name)

            if not df.empty:
                df = clean_dataframe(df, location_name)

                # Remove jobs we've already collected
                if "job_url" in df.columns:
                    df = df[~df["job_url"].isin(seen_urls)]
                    seen_urls.update(df["job_url"].tolist())

                if not df.empty:
                    new_jobs_total += len(df)
                    all_jobs = pd.concat([all_jobs, df], ignore_index=True)

            # Save checkpoint every 10 searches in case of interruption
            if search_count % 10 == 0 and not all_jobs.empty:
                save_checkpoint(all_jobs)
                tqdm.write(f"  Checkpoint saved — {len(all_jobs)} jobs so far")

            pbar.update(1)

            # Random delay to reduce the chance of getting blocked
            time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

    pbar.close()

    # ─────────────────────────────────────────
    # Save final output
    # ─────────────────────────────────────────

    if all_jobs.empty:
        print("\nNo data collected.")
        return

    # Final deduplication by job URL
    if "job_url" in all_jobs.columns:
        before = len(all_jobs)
        all_jobs = all_jobs.drop_duplicates(subset=["job_url"])
        removed = before - len(all_jobs)
        if removed:
            print(f"\nRemoved {removed} duplicate entries.")

    # Sort by most recent postings first
    if "date_posted" in all_jobs.columns:
        all_jobs = all_jobs.sort_values("date_posted", ascending=False)

    # Save to CSV with UTF-8 BOM for Excel compatibility
    all_jobs.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    # Clean up checkpoint file after successful completion
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)

    # ─────────────────────────────────────────
    # Print summary
    # ─────────────────────────────────────────

    print("\n" + "=" * 55)
    print(f"  Done! Output saved to: {OUTPUT_FILE}")
    print(f"  Total jobs collected: {len(all_jobs):,}")
    print(f"  New unique jobs: {new_jobs_total:,}")

    print("\n  Jobs by country:")
    if "country" in all_jobs.columns:
        for country, count in all_jobs["country"].value_counts().items():
            print(f"    {country:<20} {count:>5,}")

    print("\n  Top job titles:")
    if "title" in all_jobs.columns:
        for title, count in all_jobs["title"].value_counts().head(5).items():
            print(f"    {title:<35} {count:>4,}")

    print("=" * 55)


main()

  LinkedIn Tech Jobs Scraper
  30 keywords x 5 locations
  Total searches: 150


Overall Progress:   7%|▋         | 10/150 [06:57<1:38:25, 42.18s/search, location=Egypt, keyword=data analyst, total=84]

  Checkpoint saved — 94 jobs so far


Overall Progress:  13%|█▎        | 20/150 [14:19<1:29:37, 41.36s/search, location=Egypt, keyword=cybersecurity e, total=174]

  Checkpoint saved — 184 jobs so far


...

  Checkpoint saved — 1036 jobs so far


Overall Progress: 100%|██████████| 150/150 [1:48:20<00:00, 35.55s/search, location=Kuwait, keyword=game developer, total=1071]

  Checkpoint saved — 1071 jobs so far


Overall Progress: 100%|██████████| 150/150 [1:48:30<00:00, 43.41s/search, location=Kuwait, keyword=game developer, total=1071]


  Done! Output saved to: /Workspace/Users/felooamer@gmail.com/linkedin_jobs.csv
  Total jobs collected: 1,071
  New unique jobs: 1,071

  Jobs by country:
    UAE                    271
    Egypt                  260
    Saudi Arabia           243
    Qatar                  175
    Kuwait                 122

  Top job titles:
    Product Designer                      16
    DevOps Engineer                       15
    Junior Front-End Developer            14
    Data Analyst                          14
    Product Manager                       13


In [0]:
import pandas as pd

df = pd.read_csv( "/Workspace/Users/felooamer@gmail.com/linkedin_jobs.csv")
print(f"Total jobs: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
display(df.head(20))

Total jobs: 1,071
Columns: ['title', 'company', 'location', 'date_posted', 'job_type', 'job_url', 'country']


title,company,location,date_posted,job_type,job_url,country
Machine Learning Engineer,Jobgether,null,2026-05-07,fulltime,https://www.linkedin.com/jobs/view/4410447569,Saudi Arabia
"Specialist, Infrastructure (Network Admin.)",ArcelorMittal Tubular Products Al-Jubail,null,2026-05-07,fulltime,https://www.linkedin.com/jobs/view/4409243973,Saudi Arabia
Cyber Security Specialist,Avensys Consulting,null,2026-05-07,fulltime,https://www.linkedin.com/jobs/view/4411632254,Saudi Arabia
Cloud Infrastructure Engineer,Artificial Intelligence Global Company,"Eastern, Saudi Arabia",2026-05-07,fulltime,https://www.linkedin.com/jobs/view/4411446016,Saudi Arabia
DevOps Engineer I,MOZN,null,2026-05-07,fulltime,https://www.linkedin.com/jobs/view/4402621563,Saudi Arabia
"Full-Stack Software Engineer, (Forward Deployed), GPS",Scale AI,"Doha, Qatar",2026-05-07,fulltime,https://www.linkedin.com/jobs/view/4389782741,Qatar
"Staff FullStack Software Engineer, (Forward Deployed), GPS",Scale AI,"Doha, Qatar",2026-05-07,null,https://www.linkedin.com/jobs/view/4389961557,Qatar
"Senior Full-Stack Software Engineer, (Forward Deployed), GPS",Scale AI,"Doha, Qatar",2026-05-07,fulltime,https://www.linkedin.com/jobs/view/4389794336,Qatar
Vice President of Business Development - Real Estate,Confidential Government,"Riyadh, Saudi Arabia",2026-05-07,fulltime,https://www.linkedin.com/jobs/view/4409291205,Saudi Arabia
Web Designer,Armasite,"Doha, Qatar",2026-05-07,fulltime,https://www.linkedin.com/jobs/view/4409252696,Qatar
